# Colab pilot: SynthHuman dari 6 chunk lokal

Notebook ini menyalin enam subfolder chunk (sekitar 30.000 sample) dari Google Drive ke disk sementara Colab `/content`, lalu menjalankan preprocessing dan **1 epoch** depth + normals. Sel staging menampilkan progress per file dan bisa dilanjutkan selama runtime yang sama. Dataset hasil copy hilang saat runtime di-reset; data sumber dan checkpoint tetap di Drive.

Pilih GPU Colab sebelum menjalankan. Ini pilot untuk mengukur copy dan throughput lokal; belum mewakili full release.

In [ ]:
# 0. Dependency preflight
import importlib.util
missing = [name for name in ('timm', 'skimage', 'cv2', 'OpenEXR') if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError('Missing packages: ' + ', '.join(missing) + '. Run in a separate cell: %pip install timm scikit-image opencv-python-headless "OpenEXR>=3.3,<4" then restart runtime.')
print('Dependencies present: timm, scikit-image, OpenCV, OpenEXR')

In [ ]:
# 1. Runtime, imports, and pilot settings
import os, json, math, random, time, platform, re, shutil
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt
try:
    from skimage.metrics import structural_similarity as skimage_ssim
except ImportError:
    skimage_ssim = None

SEED = 42
INPUT_SIZE = (256, 256)
BATCH_SIZE, EPOCHS, NUM_WORKERS = 8, 1, 2
DATASET_MODE = 'pilot'
PILOT_SAMPLE_COUNT = 30_000  # 6 chunks x ~5,000 samples
CHUNK_SIZE = 5_000
NUM_CHUNKS_TO_STAGE = 6
CHUNK_ID_CEILING = 60
RESERVE_BYTES = 3 * 1024**3
LR, WEIGHT_DECAY, VAL_FRACTION, TEST_FRACTION = 1e-4, 1e-4, 0.15, 0.15
DEPTH_MODE = 'metric'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = torch.cuda.is_available()
print({'device':str(DEVICE),'torch':torch.__version__,'python':platform.python_version(),'input_size':INPUT_SIZE})
if torch.cuda.is_available():
    print({'cuda':torch.version.cuda,'cudnn':torch.backends.cudnn.version(),'gpu':torch.cuda.get_device_name(0),'gpu_count':torch.cuda.device_count(),'gpu_memory_GiB':round(torch.cuda.get_device_properties(0).total_memory/1024**3,2)})
else:
    print('GPU tidak terdeteksi: ubah Runtime > Change runtime type > GPU sebelum training.')

### Google Drive persistence
Colab runtimes are temporary. Mount Drive at the start of every new runtime session, then keep the dataset, manifest, pretrained-weight caches, checkpoints, training history, metrics, and Core ML exports under `MyDrive/AppleCapstoneRelighting`.

In [ ]:
# 2. Mount Drive dan tentukan source/persistent paths
from google.colab import drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/AppleCapstoneRelighting')
SOURCE_ROOT = DRIVE_ROOT / 'dataset' / 'sources' / 'SynthHuman'
LOCAL_ROOT = Path('/content/SynthHuman_6chunk_pilot')
DATA_ROOT = DRIVE_ROOT / 'dataset'
PILOT_ROOT = DATA_ROOT / 'pilot_manifests'
PILOT_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST = PILOT_ROOT / f'synthhuman_6chunks_{PILOT_SAMPLE_COUNT}_seed{SEED}.jsonl'
OUTPUT_DIR = DRIVE_ROOT / 'runs' / f'fastvit_t8_fpn_depth_normals_6chunks_{PILOT_SAMPLE_COUNT}_seed{SEED}_1ep'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_ROOT = DRIVE_ROOT / 'cache'
os.environ['TORCH_HOME'] = str(CACHE_ROOT / 'torch')
os.environ['HF_HOME'] = str(CACHE_ROOT / 'huggingface')
for folder in (CACHE_ROOT / 'torch', CACHE_ROOT / 'huggingface'):
    folder.mkdir(parents=True, exist_ok=True)
assert SOURCE_ROOT.is_dir(), f'SynthHuman source folder not found: {SOURCE_ROOT}'
print('Drive source:', SOURCE_ROOT)
print('Colab local staging:', LOCAL_ROOT)
print('Local disk free:', round(shutil.disk_usage('/content').free/1024**3,1), 'GiB')
print('Persistent manifest/checkpoints:', OUTPUT_DIR)

### Stage six Drive chunks to Colab local disk

Cell ini memilih folder yang di-hardcode (chunk_0009 sampai chunk_0014) dan mencari tiap nama langsung lewat Google Drive API. Tidak ada pemindaian semua subfolder SynthHuman atau Path.is_dir() pada folder FUSE yang padat. File di dalam enam chunk diindeks lalu diunduh ke /content dengan progress bar; file lengkap yang sudah ada bisa dipakai ulang.


In [ ]:
# 2a. Use Drive API (not Drive FUSE) to find and stage six extracted chunks
# Drive FUSE can raise Errno 5 when SynthHuman has hundreds of thousands of direct entries.
import os, re, shutil, time, threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from google.auth import default
from googleapiclient.discovery import build
from google.auth.transport.requests import AuthorizedSession

credentials, _ = default()
drive_api = build('drive', 'v3', credentials=credentials, cache_discovery=False)
FOLDER_MIME = 'application/vnd.google-apps.folder'

def list_drive_children(parent_id, folders_only=False):
    query = f"'{parent_id}' in parents and trashed = false"
    if folders_only: query += f" and mimeType = '{FOLDER_MIME}'"
    result=[]; token=None
    while True:
        page=drive_api.files().list(q=query, pageSize=1000, pageToken=token, fields='nextPageToken,files(id,name,mimeType,size)').execute()
        result.extend(page.get('files', [])); token=page.get('nextPageToken')
        if not token: return result

def find_drive_folder(parent_id, name):
    # Query this exact child name; don't enumerate every sibling folder.
    escaped_name=name.replace("'", "\\'")
    query=f"'{parent_id}' in parents and name = '{escaped_name}' and mimeType = '{FOLDER_MIME}' and trashed = false"
    matches=[]; token=None
    while True:
        page=drive_api.files().list(q=query,pageSize=100,pageToken=token,fields='nextPageToken,files(id,name,mimeType)').execute()
        matches.extend(page.get('files',[])); token=page.get('nextPageToken')
        if not token: break
    if not matches: raise FileNotFoundError(f'Drive API cannot find folder {name!r} under parent id {parent_id}')
    if len(matches)>1: print(f'Warning: {len(matches)} folders named {name}; using first result.')
    return matches[0]

# Resolve the known Drive path by IDs; never stat/list the crowded SynthHuman FUSE folder.
project_folder=find_drive_folder('root', 'AppleCapstoneRelighting')
dataset_folder=find_drive_folder(project_folder['id'], 'dataset')
sources_folder=find_drive_folder(dataset_folder['id'], 'sources')
synthhuman_folder=find_drive_folder(sources_folder['id'], 'SynthHuman')
CHUNK_NAMES=['chunk_0009','chunk_0010','chunk_0011','chunk_0012','chunk_0013','chunk_0014']
selected_chunks=[find_drive_folder(synthhuman_folder['id'],name) for name in CHUNK_NAMES]
CHUNK_DIRS=[SOURCE_ROOT/item['name'] for item in selected_chunks]  # logical Drive paths; never stat these
print('Selected hardcoded Drive chunk folders:', [item['name'] for item in selected_chunks])

MODALITY_PATTERN=re.compile(r'^(rgb|alpha|depth|normal)_(\d+)\.(png|exr)$')
def list_chunk_files(folder_id, relative_prefix=''):
    found=[]
    for item in list_drive_children(folder_id):
        relative=f'{relative_prefix}/{item["name"]}'.strip('/')
        if item['mimeType']==FOLDER_MIME:
            found.extend(list_chunk_files(item['id'], relative))
        elif MODALITY_PATTERN.fullmatch(item['name']):
            expected_ext='png' if item['name'].startswith(('rgb_', 'alpha_')) else 'exr'
            if item['name'].lower().endswith('.'+expected_ext):
                item['relative_path']=relative
                item['size_bytes']=int(item.get('size', 0))
                found.append(item)
    return found

jobs_by_chunk={}; total_bytes=0; total_files=0
for chunk in selected_chunks:
    jobs=list_chunk_files(chunk['id'])
    jobs_by_chunk[chunk['name']]=jobs
    chunk_bytes=sum(item['size_bytes'] for item in jobs)
    total_files+=len(jobs); total_bytes+=chunk_bytes
    print(f"Indexed {chunk['name']}: {len(jobs):,} modality files, {chunk_bytes/1024**3:.1f} GiB")
if total_files==0 or any(not jobs for jobs in jobs_by_chunk.values()):
    raise RuntimeError('At least one selected chunk has no recognized modality files. Check its extracted contents.')
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
free_bytes=shutil.disk_usage('/content').free
print(f'Total files={total_files:,}; estimated bytes={total_bytes/1024**3:.1f} GiB; /content free={free_bytes/1024**3:.1f} GiB')
if free_bytes < total_bytes+RESERVE_BYTES:
    raise OSError(f'Insufficient Colab local disk: need about {total_bytes/1024**3:.1f} GiB plus reserve.')

thread_local=threading.local()
def get_authorized_session():
    if not hasattr(thread_local, 'session'): thread_local.session=AuthorizedSession(credentials)
    return thread_local.session
def download_one(chunk_name, item):
    target=LOCAL_ROOT/chunk_name/item['relative_path']
    expected=item['size_bytes']; target.parent.mkdir(parents=True, exist_ok=True)
    if expected and target.is_file() and target.stat().st_size==expected: return expected, False
    temporary=target.with_name(target.name+'.part')
    url=f"https://www.googleapis.com/drive/v3/files/{item['id']}"
    error=None
    for attempt in range(5):
        try:
            with get_authorized_session().get(url, params={'alt':'media'}, stream=True, timeout=(30,300)) as response:
                response.raise_for_status()
                with temporary.open('wb') as output:
                    for block in response.iter_content(chunk_size=4*1024**2):
                        if block: output.write(block)
            actual=temporary.stat().st_size
            if expected and actual!=expected: raise IOError(f"size mismatch for {item['name']}: expected {expected}, got {actual}")
            os.replace(temporary,target)
            return actual, True
        except Exception as exc:
            error=exc; temporary.unlink(missing_ok=True)
            if attempt<4: time.sleep(min(2**attempt,16))
    raise RuntimeError(f"Could not download Drive file {item['name']} ({item['id']}): {error}")

started=time.perf_counter(); copied_files=resumed_files=0
with ThreadPoolExecutor(max_workers=8) as pool:
    for chunk_pos,(chunk_name,jobs) in enumerate(jobs_by_chunk.items(),1):
        chunk_bytes=sum(item['size_bytes'] for item in jobs); chunk_start=time.perf_counter()
        print(f'\nDownload chunk {chunk_pos}/{len(selected_chunks)} · {chunk_name}: {len(jobs):,} files')
        with tqdm(total=chunk_bytes,unit='B',unit_scale=True,desc=f'{chunk_pos}/{len(selected_chunks)} {chunk_name}',dynamic_ncols=True) as progress:
            for batch_start in range(0,len(jobs),512):
                futures=[pool.submit(download_one,chunk_name,item) for item in jobs[batch_start:batch_start+512]]
                for future in as_completed(futures):
                    size,did_copy=future.result(); progress.update(size)
                    if did_copy: copied_files+=1
                    else: resumed_files+=1
                    progress.set_postfix(files=f'{copied_files+resumed_files:,}/{total_files:,}',resumed=resumed_files,refresh=False)
        print(f'Chunk {chunk_name} finished in {(time.perf_counter()-chunk_start)/60:.1f} min')
print(f'All six chunks staged in {(time.perf_counter()-started)/60:.1f} min; downloaded={copied_files:,}, already present={resumed_files:,}')


In [ ]:
# 3. Index only the six local chunks and create a deterministic labeled manifest
from collections import defaultdict
MODALITIES = {'rgb':'png', 'alpha':'png', 'depth':'exr', 'normal':'exr'}
groups = defaultdict(dict); ambiguous = set()
for chunk_dir in CHUNK_DIRS:
    local_chunk = LOCAL_ROOT / chunk_dir.relative_to(SOURCE_ROOT)
    for dirpath, _dirnames, filenames in os.walk(local_chunk):
        for filename in filenames:
            match = MODALITY_PATTERN.fullmatch(filename)
            if not match: continue
            kind, raw_id, extension = match.groups()
            if extension != MODALITIES[kind]: continue
            sample_id = int(raw_id); path = Path(dirpath) / filename
            if kind in groups[sample_id] and groups[sample_id][kind] != path:
                ambiguous.add(sample_id)
            else:
                groups[sample_id][kind] = path
complete_ids = sorted(sample_id for sample_id, group in groups.items() if set(MODALITIES) <= group.keys() and sample_id not in ambiguous)
print(f'Local inventory: {len(complete_ids):,} complete; {len(ambiguous):,} ambiguous; {len(groups)-len(complete_ids)-len(ambiguous):,} incomplete IDs')
if len(complete_ids) < PILOT_SAMPLE_COUNT:
    raise ValueError(f'Only {len(complete_ids):,} complete samples found in six staged chunks; need {PILOT_SAMPLE_COUNT:,}. Check chunk completeness or lower PILOT_SAMPLE_COUNT.')
selected_ids = sorted(random.Random(SEED).sample(complete_ids, PILOT_SAMPLE_COUNT))
rows = []
for sample_id in selected_ids:
    group = groups[sample_id]
    rows.append({'image':str(group['rgb']), 'mask':str(group['alpha']), 'depth':str(group['depth']), 'normal':str(group['normal']), 'subject_id':f'sample_{sample_id:07d}', 'label_source':'SynthHuman/DAViD renderer', 'depth_units':'m', 'depth_scale_to_m':0.01, 'normal_convention':'camera_xyz_signed'})
MANIFEST.parent.mkdir(parents=True, exist_ok=True)
tmp_manifest = MANIFEST.with_suffix('.jsonl.tmp')
tmp_manifest.write_text(''.join(json.dumps(row)+'\n' for row in rows), encoding='utf-8')
tmp_manifest.replace(MANIFEST)
print(f'Selected {len(rows):,} complete samples; manifest saved to Drive: {MANIFEST}')
print('This local `/content` copy is temporary; rerun the staging cell after a fresh runtime.')


### Manifest validation and split

Manifest memuat path lokal Colab untuk RGB, alpha mask, depth EXR, dan normal EXR. Pemisahan train/validation/test dilakukan per sample ID (bukan per identitas orang, karena metadata identitas tidak tersedia).

In [ ]:
# 4. Read and validate the automatically generated manifest
def load_manifest(path,verify_paths=True):
    if not path.is_file(): raise FileNotFoundError(f'Manifest missing: {path}. Run the dataset manifest cell first.')
    rows=[json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    required={'image','depth','normal','mask','subject_id','label_source','depth_units','normal_convention'}
    if not rows: raise ValueError('Manifest contains no rows')
    for i,row in enumerate(rows):
        missing=required-row.keys()
        if missing: raise ValueError(f'row {i} missing fields: {sorted(missing)}')
        for key in ('image','depth','normal','mask'):
            p=Path(row[key]); p=p if p.is_absolute() else DATA_ROOT/p
            if verify_paths and not p.is_file(): raise FileNotFoundError(f'row {i} {key}: {p}')
            row[key]=str(p)
    return rows
ALL_ROWS=load_manifest(MANIFEST,verify_paths=True)
if DATASET_MODE == 'full':
    rows=ALL_ROWS
    RUN_MANIFEST=MANIFEST
elif DATASET_MODE == 'pilot':
    if len(ALL_ROWS)<PILOT_SAMPLE_COUNT:
        raise ValueError(f'Pilot requires {PILOT_SAMPLE_COUNT} complete labeled samples, but manifest has {len(ALL_ROWS)}.')
    pilot_rng=random.Random(SEED)
    rows=sorted(pilot_rng.sample(ALL_ROWS,PILOT_SAMPLE_COUNT),key=lambda r:r['subject_id'])
    RUN_MANIFEST=DATA_ROOT/'pilot_manifests'/f'depth_pilot_{PILOT_SAMPLE_COUNT}_seed{SEED}.jsonl'
    RUN_MANIFEST.parent.mkdir(parents=True,exist_ok=True)
    pilot_tmp=RUN_MANIFEST.with_suffix('.jsonl.tmp'); pilot_tmp.write_text(''.join(json.dumps(r)+'\n' for r in rows)); pilot_tmp.replace(RUN_MANIFEST)
else:
    raise ValueError("DATASET_MODE must be 'full' or 'pilot'")
print(f'Run mode={DATASET_MODE}; samples={len(rows):,}; active labels: depth + normals + mask')
print('Run manifest:',RUN_MANIFEST)

### Identity split
Generate the fixed train/validation/test split before augmentations. The saved `split.json` makes the partition reviewable and reproducible.

In [ ]:
# 5. Split by sample ID (source has no verified person identity IDs)
def split_by_subject(rows,seed=SEED):
    ids=sorted({r['subject_id'] for r in rows})
    rng=random.Random(seed); rng.shuffle(ids)
    n_test=max(1,round(len(ids)*TEST_FRACTION)); n_val=max(1,round(len(ids)*VAL_FRACTION))
    test_ids=set(ids[:n_test]); val_ids=set(ids[n_test:n_test+n_val]); train_ids=set(ids)-test_ids-val_ids
    parts={name:[r for r in rows if r['subject_id'] in subset] for name,subset in [('train',train_ids),('val',val_ids),('test',test_ids)]}
    for name, group in parts.items():
        for row in group: row['partition']=name
    split_info={'seed':seed,'split_unit':'sample_id (not verified person identity)','train_subjects':sorted(train_ids),'val_subjects':sorted(val_ids),'test_subjects':sorted(test_ids)}
    (OUTPUT_DIR/'split.json').write_text(json.dumps(split_info,indent=2))
    return parts
parts=split_by_subject(rows)
print({k:len(v) for k,v in parts.items()})
print('Evaluation limitation: split is sample-disjoint; this release does not expose verified identity IDs.')

### Image and label preprocessing
This cell defines loading/resizing and paired transforms only. It does not download data, change the split, or start training.

In [ ]:
# 6. Raw local-file reader: EXR depth/normals decoded during preprocessing
import random
from pathlib import Path
from torch.utils.data import Dataset
from PIL import Image
def _resize_array(array, size, is_mask=False):
    array = np.asarray(array)
    if array.ndim == 2:
        array = array[..., None]
    mode = Image.Resampling.NEAREST if is_mask or array.shape[-1] == 1 else Image.Resampling.BILINEAR
    planes = [np.asarray(Image.fromarray(array[..., channel].astype(np.float32)).resize((size[1], size[0]), mode), dtype=np.float32) for channel in range(array.shape[-1])]
    return np.stack(planes, axis=-1)


def _read_exr(path: Path, kind: str):
    with path.open("rb") as handle:
        prefix = handle.read(16)
    if prefix[:4] != b"\x76\x2f\x31\x01":
        raise ValueError(f"{path} has no valid EXR magic header (first16={prefix.hex()})")
    try:
        import OpenEXR
    except ImportError as error:
        raise RuntimeError('Install OpenEXR in the DGX environment: pip install "OpenEXR>=3.3,<4"') from error
    if not hasattr(OpenEXR, "File"):
        raise RuntimeError("OpenEXR >=3.3 is required for the named-channel decoder")
    with OpenEXR.File(str(path), separate_channels=True) as exr:
        if len(exr.parts) != 1:
            raise ValueError(f"Multipart EXR is unsupported: {path}")
        channels = exr.channels()
        lookup = {name.upper(): name for name in channels}
        if kind == "normal":
            if {"R", "G", "B"} <= lookup.keys():
                keys = ("R", "G", "B")
            elif {"X", "Y", "Z"} <= lookup.keys():
                keys = ("X", "Y", "Z")
            else:
                raise ValueError(f"Normal EXR needs RGB or XYZ channels, found {sorted(channels)}: {path}")
        elif "Z" in lookup:
            keys = ("Z",)
        elif len(channels) == 1:
            keys = (next(iter(channels)).upper(),)
        elif "R" in lookup:
            keys = ("R",)
        else:
            raise ValueError(f"Ambiguous depth EXR channels {sorted(channels)}: {path}")
        planes = []
        for key in keys:
            pixels = channels[lookup[key]].pixels
            if pixels.ndim != 2 or pixels.dtype.kind not in "fiu":
                raise ValueError(f"Invalid EXR channel {key}: {path}")
            planes.append(np.array(pixels, dtype=np.float32, copy=True))
        return np.stack(planes, axis=-1)


def _read_rgb(path: Path, size):
    with Image.open(path) as image:
        image = image.convert("RGB").resize((size[1], size[0]), Image.Resampling.BILINEAR)
        return np.asarray(image, dtype=np.float32) / 255.0


class RawSynthHumanDataset(Dataset):
    def __init__(self, rows, input_size=(256, 256), augment=False):
        self.rows = rows
        self.input_size = tuple(input_size)
        self.augment = augment

    def __len__(self):
        return len(self.rows)

    def close(self):
        return None

    def __getitem__(self, index):
        augmentation_seed = None
        if isinstance(index, tuple):
            index, augmentation_seed = index
        row = self.rows[index]
        image = _read_rgb(Path(row["image"]), self.input_size)
        raw_depth = _read_exr(Path(row["depth"]), "depth")
        normal = _read_exr(Path(row["normal"]), "normal")
        with Image.open(row["mask"]) as alpha_image:
            alpha = np.asarray(alpha_image.convert("L"))
        mask = _resize_array((alpha > 0).astype(np.uint8) * 255, self.input_size, True)[..., 0] > 0
        raw_depth = _resize_array(raw_depth.astype(np.float32), self.input_size)[..., 0]
        normal = _resize_array(normal.astype(np.float32), self.input_size)
        if normal.shape[-1] != 3:
            raise ValueError(f"Expected 3-channel normals at {row['normal']}; found {normal.shape}")
        valid_depth = np.isfinite(raw_depth) & (raw_depth > 0) & (raw_depth < 1000) & (~np.isclose(raw_depth, 65504.0, rtol=0.0, atol=1.0))
        depth = raw_depth[..., None] * float(row.get("depth_scale_to_m", 0.01))
        mask &= valid_depth & np.isfinite(normal).all(axis=-1)
        depth = np.nan_to_num(depth, nan=0.0, posinf=0.0, neginf=0.0)
        normal = np.nan_to_num(normal, nan=0.0, posinf=0.0, neginf=0.0)
        if mask.sum() == 0:
            raise ValueError(f"No valid labeled pixels for {row['subject_id']}")
        rng = random.Random(augmentation_seed) if augmentation_seed is not None else random
        if self.augment and rng.random() < 0.5:
            image, depth, normal, mask = [np.flip(array, axis=1).copy() for array in (image, depth, normal, mask)]
            normal[..., 0] *= -1
        image = (image - np.array([0.485, 0.456, 0.406], np.float32)) / np.array([0.229, 0.224, 0.225], np.float32)
        tensors = [torch.from_numpy(array.copy()).permute(2, 0, 1).float() for array in (image, depth, normal)]
        tensors.append(torch.from_numpy(mask[None].astype(np.float32)))
        tensors[2] = F.normalize(tensors[2], dim=0, eps=1e-6)
        tensors[3] *= (torch.linalg.vector_norm(tensors[2], dim=0, keepdim=True) > 0).float()
        return tuple(tensors)


### What preprocessing produces
Each sample returns four tensors: ImageNet-normalized RGB `(3,256,256)`; metric depth `(1,256,256)` in metres (source EXR centimetres × 0.01); unit normals `(3,256,256)` in camera XYZ; and valid-pixel mask `(1,256,256)`. Invalid/non-finite depth, the EXR sentinel, and zero-length normals are excluded from the mask. The model returns full-resolution depth `(B,1,256,256)` and normals `(B,3,256,256)`. The loader audit prints tensor shapes/ranges and shows one sample before training.


### DataLoader and sample audit
Run this after the previous cells. Inspect the displayed example and ranges before constructing the model.

In [ ]:
# 7. Build loaders and audit one real sample
NUM_WORKERS=2  # Local raw files; worker processes can decode independent samples.
datasets={k:RawSynthHumanDataset(v, input_size=INPUT_SIZE, augment=(k=='train')) for k,v in parts.items()}
loaders={k:DataLoader(v,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'),drop_last=False) for k,v in datasets.items()}
x,d,n,m=datasets['train'][0]
valid_depth_values=d[0][m[0]>0]
print('Preprocessing output shapes:', {'rgb':tuple(x.shape),'depth_m':tuple(d.shape),'normal_unit':tuple(n.shape),'valid_mask':tuple(m.shape)})
print('RGB normalized channel mean/std:',x.mean((1,2)).tolist(),x.std((1,2)).tolist())
print('valid depth metres p01/p50/p99/max',*[float(torch.quantile(valid_depth_values,q)) for q in (0.01,0.50,0.99)],float(valid_depth_values.max()),'normal range',float(n.min()),float(n.max()),'mask fraction',float(m.mean()))
assert m.sum()>0,'Empty valid mask'
fig,ax=plt.subplots(1,4,figsize=(15,4)); ax[0].imshow(np.clip(x.permute(1,2,0).numpy()*[.229,.224,.225]+[.485,.456,.406],0,1)); ax[0].set_title('RGB'); ax[1].imshow(d[0],cmap='viridis'); ax[1].set_title('Depth (m)'); ax[2].imshow(n.permute(1,2,0)*.5+.5); ax[2].set_title('Normals'); ax[3].imshow(m[0],cmap='gray'); ax[3].set_title('Valid mask'); [z.axis('off') for z in ax]; plt.show()

In [ ]:
# 8. Shared multi-task architecture with a selectable mobile backbone
# Default preserves the project baseline (FastViT-T8). MobileNetV3-Large is an ablation only.
import timm
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

BACKBONE = 'fastvit_t8.apple_in1k'  # compare candidates via the same train/eval protocol

class FastViTEncoder(nn.Module):
    def __init__(self,pretrained=True):
        super().__init__()
        self.net=timm.create_model('fastvit_t8.apple_in1k',pretrained=pretrained,features_only=True)
        self.channels=tuple(self.net.feature_info.channels())
    def forward(self,x): return self.net(x)

class MobileNetV3Encoder(nn.Module):
    """Experimental alternative; exposes stride-4/8/16/32 features."""
    def __init__(self,pretrained=True):
        super().__init__()
        weights=MobileNet_V3_Large_Weights.DEFAULT if pretrained else None
        self.features=mobilenet_v3_large(weights=weights).features
        self.tap_indices=(3,6,12,15)
        self.channels=(24,40,112,160)
    def forward(self,x):
        outputs=[]
        for idx,layer in enumerate(self.features):
            x=layer(x)
            if idx in self.tap_indices: outputs.append(x)
        if len(outputs)!=len(self.channels): raise RuntimeError('Unexpected MobileNetV3 feature taps')
        return outputs

class DepthwiseSeparableBlock(nn.Module):
    def __init__(self,channels):
        super().__init__()
        self.depthwise=nn.Conv2d(channels,channels,3,padding=1,groups=channels,bias=False)
        self.depthwise_bn=nn.BatchNorm2d(channels)
        self.pointwise=nn.Conv2d(channels,channels,1,bias=False)
        self.pointwise_bn=nn.BatchNorm2d(channels)
    def forward(self,x):
        x=F.relu(self.depthwise_bn(self.depthwise(x)),inplace=True)
        return F.relu(self.pointwise_bn(self.pointwise(x)),inplace=True)

class CompactFPN(nn.Module):
    def __init__(self,channels,width=48):
        super().__init__()
        self.lateral=nn.ModuleList([nn.Conv2d(c,width,1) for c in channels])
        self.smooth=nn.ModuleList([DepthwiseSeparableBlock(width) for _ in channels])
    def forward(self,features):
        y=self.smooth[-1](self.lateral[-1](features[-1]))
        for idx in range(len(features)-2,-1,-1):
            y=F.interpolate(y,size=features[idx].shape[-2:],mode='bilinear',align_corners=False)
            y=self.smooth[idx](y+self.lateral[idx](features[idx]))
        return y

class MobileTaskHead(nn.Module):
    def __init__(self,width,out_channels):
        super().__init__(); self.refine=DepthwiseSeparableBlock(width); self.predict=nn.Conv2d(width,out_channels,1)
    def forward(self,x,out_size):
        return F.interpolate(self.predict(self.refine(x)),size=out_size,mode='bilinear',align_corners=False)

class PortraitGeometryModel(nn.Module):
    """One RGB crop -> metric depth and unit surface normals."""
    def __init__(self,backbone=BACKBONE,pretrained=True,fpn_width=48):
        super().__init__()
        if backbone=='fastvit_t8.apple_in1k': self.encoder=FastViTEncoder(pretrained=pretrained)
        elif backbone=='mobilenet_v3_large': self.encoder=MobileNetV3Encoder(pretrained=pretrained)
        else: raise ValueError(f'Unsupported backbone: {backbone}')
        self.backbone_name=backbone
        self.neck=CompactFPN(self.encoder.channels,width=fpn_width)
        self.depth_head=MobileTaskHead(fpn_width,1)
        self.normal_head=MobileTaskHead(fpn_width,3)
    def forward(self,x):
        out_size=x.shape[-2:]; fused=self.neck(self.encoder(x))
        depth=F.softplus(self.depth_head(fused,out_size))
        normals=F.normalize(self.normal_head(fused,out_size),p=2,dim=1,eps=1e-6)
        return depth,normals

model=PortraitGeometryModel(backbone=BACKBONE,pretrained=True).to(DEVICE)
nparams=sum(p.numel() for p in model.parameters())
with torch.no_grad(): outputs=model(torch.zeros(2,3,*INPUT_SIZE,device=DEVICE))
assert [tuple(t.shape) for t in outputs]==[(2,1,*INPUT_SIZE),(2,3,*INPUT_SIZE)]
assert all(torch.isfinite(t).all() for t in outputs)
print(f'Backbone={BACKBONE}; total parameters={nparams/1e6:.2f}M; FPN width=48; feature channels={model.encoder.channels}')


In [ ]:
# 9. Masked depth and normal losses/metrics
class GeometryObjective(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_vars=nn.Parameter(torch.zeros(2))
    @staticmethod
    def masked_mean(value,mask):
        return (value*mask).sum()/mask.sum().clamp_min(1.)
    def forward(self,pd,td,pn,tn,mask):
        if DEPTH_MODE=='relative':
            pd0=pd.clamp_min(1e-5).log(); td0=td.clamp_min(1e-5).log()
            for b in range(pd.shape[0]):
                valid=mask[b,0].bool()
                if not valid.any(): raise ValueError('A sample has an empty valid mask')
                pd0[b]-=torch.median(pd0[b,0][valid]); td0[b]-=torch.median(td0[b,0][valid])
            depth_loss=self.masked_mean(F.smooth_l1_loss(pd0,td0,reduction='none'),mask)
        else:
            depth_loss=self.masked_mean(F.smooth_l1_loss(pd,td,reduction='none'),mask)
        cosine=(F.normalize(pn,dim=1,eps=1e-6)*F.normalize(tn,dim=1,eps=1e-6)).sum(1,keepdim=True).clamp(-1,1)
        normal_loss=self.masked_mean(1-cosine,mask)
        losses=(depth_loss,normal_loss)
        total=sum(torch.exp(-self.log_vars[j])*loss+self.log_vars[j] for j,loss in enumerate(losses))
        return total,{'depth_loss':depth_loss.detach(),'normal_loss':normal_loss.detach()}

@torch.no_grad()
def evaluate(model,loader):
    model.eval(); sums={}; counts={}
    for x,d,n,m in loader:
        x,d,n,m=[z.to(DEVICE,non_blocking=True) for z in (x,d,n,m)]
        pd,pn=model(x); valid=m[:,0].bool()
        dv=(pd[:,0]-d[:,0])[valid]
        result={'depth_mae_m':dv.abs().mean(),'depth_rmse_m':dv.square().mean().sqrt()}
        p_rel=pd[:,0].clamp_min(1e-5).log(); t_rel=d[:,0].clamp_min(1e-5).log()
        for b in range(x.shape[0]):
            mb=valid[b]
            if not mb.any(): raise ValueError('Evaluation sample has empty mask')
            p_rel[b]-=torch.median(p_rel[b][mb]); t_rel[b]-=torch.median(t_rel[b][mb])
        rr=(p_rel-t_rel)[valid]
        result['depth_log_rmse_median_aligned']=rr.square().mean().sqrt()
        dot=(F.normalize(pn,dim=1)*F.normalize(n,dim=1)).sum(1).clamp(-1,1)
        angle=torch.rad2deg(torch.acos(dot[valid]))
        result.update({'normal_mean_deg':angle.mean(),'normal_median_deg':angle.median(),'normal_within_11_25_deg':(angle<11.25).float().mean()})
        for key,value in result.items():
            sums[key]=sums.get(key,0.)+float(value); counts[key]=counts.get(key,0)+1
    if not counts: raise ValueError('Evaluation partition is empty')
    return {key:sums[key]/counts[key] for key in sums}

objective=GeometryObjective().to(DEVICE)
optimizer=torch.optim.AdamW(list(model.parameters())+list(objective.parameters()),lr=LR,weight_decay=WEIGHT_DECAY)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='min',factor=.5,patience=3)
scaler=torch.amp.GradScaler(device='cuda',enabled=(DEVICE.type=='cuda'))
print('Active supervised heads: depth + normals.')


In [ ]:
# 10. Timed training with resumable mid-epoch checkpoints on Drive
CHECKPOINT=OUTPUT_DIR/'best_checkpoint.pt'
LAST=OUTPUT_DIR/'last_checkpoint.pt'
HISTORY_PATH=OUTPUT_DIR/'history.json'
PROGRESS_PATH=OUTPUT_DIR/'progress.json'
CHECKPOINT_EVERY_MINUTES=10  # Full-epoch runs can outlast a Colab session; resume from Drive checkpoint.
PROGRESS_EVERY_STEPS=25  # Persist a compact status record every 25 iterations; tqdm updates every iteration.
print('Persistent checkpoints/history location:', OUTPUT_DIR)
partition_sizes={name:len(group) for name,group in parts.items()}
steps_per_epoch=math.ceil(partition_sizes['train']/BATCH_SIZE)
print(f'Run plan: total={len(rows):,}; split={partition_sizes}; batch={BATCH_SIZE}; epochs={EPOCHS}; train iterations/epoch={steps_per_epoch:,}; max train iterations={steps_per_epoch*EPOCHS:,}')
if any(size==0 for size in partition_sizes.values()):
    raise RuntimeError(f'Cannot train: train/validation/test split is incomplete: {partition_sizes}. Download a full chunk and rerun manifest/split/loader cells first.')

def save_checkpoint(payload,path):
    temp_path=path.with_name(path.name+'.tmp')
    torch.save(payload,temp_path)
    temp_path.replace(path)

def checkpoint_state(epoch,batch_in_epoch,epoch_order,score_value,train_totals=None,task_totals=None,train_seconds_elapsed=0.0):
    return {'epoch':epoch,'batch_in_epoch':batch_in_epoch,'epoch_order':epoch_order,'partial_train_totals':train_totals or [],'partial_task_totals':task_totals or {},'train_seconds_elapsed':train_seconds_elapsed,'model':model.state_dict(),'objective':objective.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'best_score':score_value,'history':history,'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,'config':{'seed':SEED,'input_size':INPUT_SIZE,'depth_mode':DEPTH_MODE,'backbone':BACKBONE,'fpn_width':48,'targets':['depth','normals'],'data_io_mode':'colab_local_chunks'}}

history=[]; start_epoch=0; resume_batch=0; resume_order=None; best_score=float('inf'); resume_train_totals=[]; resume_task_totals={}; resume_train_seconds=0.0
if LAST.exists():
    ckpt=torch.load(LAST,map_location=DEVICE,weights_only=False)
    model.load_state_dict(ckpt['model']); objective.load_state_dict(ckpt['objective']); optimizer.load_state_dict(ckpt['optimizer']); scheduler.load_state_dict(ckpt['scheduler']); scaler.load_state_dict(ckpt['scaler'])
    start_epoch=ckpt['epoch']; resume_batch=ckpt.get('batch_in_epoch',0); resume_order=ckpt.get('epoch_order'); best_score=ckpt.get('best_score',float('inf')); history=ckpt.get('history',[]); resume_train_totals=ckpt.get('partial_train_totals',[]); resume_task_totals=ckpt.get('partial_task_totals',{}); resume_train_seconds=ckpt.get('train_seconds_elapsed',0.0)
    random.setstate(ckpt['python_rng']); np.random.set_state(ckpt['numpy_rng']); torch.set_rng_state(ckpt['torch_rng'])
    if torch.cuda.is_available() and ckpt.get('cuda_rng') is not None: torch.cuda.set_rng_state_all(ckpt['cuda_rng'])
    print(f'Resuming epoch {start_epoch+1}, after {resume_batch} completed batches.')

for epoch in range(start_epoch,EPOCHS):
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    epoch_start=time.perf_counter()
    if DEVICE.type=='cuda': torch.cuda.reset_peak_memory_stats(DEVICE)
    model.train(); objective.train()
    if epoch==start_epoch and resume_batch>0:
        train_totals=list(resume_train_totals); task_totals={k:list(v) for k,v in resume_task_totals.items()}
    else:
        train_totals=[]; task_totals={'depth_loss':[],'normal_loss':[]}
    if epoch==start_epoch and resume_order is not None:
        epoch_order=resume_order
    else:
        epoch_order=torch.randperm(len(datasets['train']),generator=torch.Generator().manual_seed(SEED+epoch)).tolist()
        resume_batch=0
    start_sample=resume_batch*BATCH_SIZE
    remaining_indices=epoch_order[start_sample:]
    epoch_loader=DataLoader(datasets['train'],batch_size=BATCH_SIZE,sampler=remaining_indices,num_workers=NUM_WORKERS,pin_memory=(DEVICE.type=='cuda'),drop_last=False,generator=torch.Generator().manual_seed(SEED+10000+epoch))
    prior_train_seconds=resume_train_seconds if epoch==start_epoch else 0.0
    last_checkpoint_time=time.monotonic()
    last_batch_end=time.perf_counter()
    session_data_wait_seconds=0.0
    from tqdm.auto import tqdm
    full_batches=(len(datasets['train'])+BATCH_SIZE-1)//BATCH_SIZE
    progress_bar=tqdm(epoch_loader,total=full_batches,initial=resume_batch,desc=f'Epoch {epoch+1}/{EPOCHS}',unit='batch',dynamic_ncols=True)
    for batch_idx,(x,d,n,m) in enumerate(progress_bar,start=resume_batch):
        data_wait_seconds=time.perf_counter()-last_batch_end
        session_data_wait_seconds+=data_wait_seconds
        x,d,n,m=[z.to(DEVICE,non_blocking=True) for z in (x,d,n,m)]
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type,dtype=(torch.float16 if DEVICE.type=='cuda' else torch.bfloat16),enabled=(DEVICE.type=='cuda')):
            pd,pn=model(x); loss,parts=objective(pd,d,pn,n,m)
        if not torch.isfinite(loss): raise FloatingPointError(f'Non-finite training loss at epoch {epoch}')
        scaler.scale(loss).backward(); scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(list(model.parameters())+list(objective.parameters()),1.0); scaler.step(optimizer); scaler.update()
        train_totals.append(float(loss.detach()))
        for k,v in parts.items(): task_totals[k].append(float(v))
        completed_batches=batch_idx+1
        mem_gib=(torch.cuda.memory_allocated(DEVICE)/1024**3) if DEVICE.type=='cuda' else 0.0
        progress_bar.set_postfix(iteration=f'{completed_batches}/{full_batches}',loss=f'{float(loss.detach()):.4f}',depth=f'{float(parts.get("depth_loss", torch.tensor(float("nan")))):.4f}',normal=f'{float(parts.get("normal_loss", torch.tensor(float("nan")))):.4f}',gpu_mem=f'{mem_gib:.1f}GiB',local_samples=len(rows),data_s=f'{data_wait_seconds:.2f}',refresh=False)
        if completed_batches%PROGRESS_EVERY_STEPS==0:
            progress={'epoch':epoch+1,'epochs_total':EPOCHS,'batches_complete':completed_batches,'batches_total':(len(datasets['train'])+BATCH_SIZE-1)//BATCH_SIZE,'latest_train_loss':float(loss.detach()),'last_update_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}
            progress_tmp=PROGRESS_PATH.with_name(PROGRESS_PATH.name+'.tmp'); progress_tmp.write_text(json.dumps(progress,indent=2)); progress_tmp.replace(PROGRESS_PATH)
            print(f"epoch {epoch+1:03d}/{EPOCHS} batch {completed_batches}/{progress['batches_total']} loss={float(loss.detach()):.4f}")
        if time.monotonic()-last_checkpoint_time>=CHECKPOINT_EVERY_MINUTES*60:
            if DEVICE.type=='cuda': torch.cuda.synchronize()
            save_checkpoint(checkpoint_state(epoch,completed_batches,epoch_order,best_score,train_totals,task_totals,prior_train_seconds+time.perf_counter()-epoch_start),LAST)
            last_checkpoint_time=time.monotonic()
            print(f'Saved resumable checkpoint after epoch {epoch+1}, batch {completed_batches}.')
        last_batch_end=time.perf_counter()
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    train_seconds=prior_train_seconds+time.perf_counter()-epoch_start
    val_start=time.perf_counter()
    val_metrics=evaluate(model,loaders['val'])
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    val_seconds=time.perf_counter()-val_start
    preview_dir=OUTPUT_DIR/'epoch_outputs'; preview_dir.mkdir(parents=True,exist_ok=True)
    preview_x,preview_d,preview_n,preview_m=next(iter(loaders['val']))
    model.eval()
    with torch.inference_mode():
        preview_pd,preview_pn=model(preview_x[:1].to(DEVICE))
    # Persist one reproducible per-epoch matrix sample (depth + XYZ normals + GT + mask); full predictions are represented by aggregate metrics.
    preview_path=preview_dir/f'epoch_{epoch+1:03d}_sample.npz'
    np.savez_compressed(preview_path,
        depth_pred_m=preview_pd[0,0].float().cpu().numpy(),
        normal_pred_xyz=preview_pn[0].float().cpu().numpy(),
        depth_gt_m=preview_d[0,0].numpy(),
        normal_gt_xyz=preview_n[0].numpy(),
        valid_mask=preview_m[0,0].numpy())
    output_summary={'depth_pred_min_m':float(preview_pd.min()),'depth_pred_median_m':float(preview_pd.median()),'depth_pred_max_m':float(preview_pd.max()),'normal_pred_norm_mean':float(torch.linalg.vector_norm(preview_pn,dim=1).mean()),'matrix_file':str(preview_path)}
    epoch_seconds=time.perf_counter()-epoch_start
    samples_per_second=len(datasets['train'])/max(train_seconds,1e-6)
    eta_seconds=max(EPOCHS-epoch-1,0)*epoch_seconds
    score=(val_metrics['depth_rmse_m']/0.1)+(val_metrics['normal_mean_deg']/30.0)
    scheduler.step(score)
    peak_gpu_memory_gib=(torch.cuda.max_memory_allocated(DEVICE)/1024**3) if DEVICE.type=='cuda' else 0.0
    record={'epoch':epoch+1,'session_data_wait_seconds':session_data_wait_seconds,'peak_gpu_memory_gib':peak_gpu_memory_gib,'train_loss':float(np.mean(train_totals)),**{k:float(np.mean(v)) for k,v in task_totals.items()},'val':val_metrics,'selection_score':score,'train_seconds':train_seconds,'validation_seconds':val_seconds,'epoch_seconds':epoch_seconds,'train_samples_per_second':samples_per_second,'eta_seconds':eta_seconds,'lr':optimizer.param_groups[0]['lr'],'task_sigmas':torch.exp(objective.log_vars.detach()).cpu().tolist(),'output_summary':output_summary,'steps_per_epoch':full_batches,'train_samples':len(datasets['train']),'validation_samples':len(datasets['val'])}
    history.append(record)
    history_tmp=HISTORY_PATH.with_name(HISTORY_PATH.name+'.tmp'); history_tmp.write_text(json.dumps(history,indent=2)); history_tmp.replace(HISTORY_PATH)
    completed_state=checkpoint_state(epoch+1,0,None,min(best_score,score),[],{},0.0)
    save_checkpoint(completed_state,LAST)
    if score<best_score:
        best_score=score; completed_state['best_score']=best_score; completed_state['best_epoch']=epoch+1; save_checkpoint(completed_state,CHECKPOINT)
    progress={'epoch':epoch+1,'epochs_total':EPOCHS,'status':'epoch_complete','train_seconds':train_seconds,'validation_seconds':val_seconds,'epoch_seconds':epoch_seconds,'train_samples_per_second':samples_per_second,'eta_seconds':eta_seconds,'preview_matrix':str(preview_path)}
    progress_tmp=PROGRESS_PATH.with_name(PROGRESS_PATH.name+'.tmp'); progress_tmp.write_text(json.dumps(progress,indent=2)); progress_tmp.replace(PROGRESS_PATH)
    resume_batch=0; resume_order=None; resume_train_totals=[]; resume_task_totals={}; resume_train_seconds=0.0
    print(f"epoch {epoch+1:03d}/{EPOCHS} iterations={full_batches:,} train={record['train_loss']:.4f} val depth(RMSE m)={val_metrics['depth_rmse_m']:.4f} normal(mean°)={val_metrics['normal_mean_deg']:.2f} score={score:.4f} train={train_seconds/60:.1f}m val={val_seconds/60:.1f}m rate={samples_per_second:.1f} samples/s ETA={eta_seconds/60:.1f}m")

if not CHECKPOINT.exists() and LAST.exists():
    save_checkpoint(torch.load(LAST,map_location=DEVICE,weights_only=False),CHECKPOINT)
if not CHECKPOINT.exists(): raise RuntimeError('No checkpoint found. Complete at least one epoch first.')
best=torch.load(CHECKPOINT,map_location=DEVICE,weights_only=False); model.load_state_dict(best['model'])
print('Best checkpoint epoch:',best.get('best_epoch',best['epoch']),'selection score:',best['best_score'])

### Reading training progress

The six chunks provide up to 30,000 complete samples, split approximately 21,000 train / 4,500 validation / 4,500 test. This pilot runs one epoch (about 2,625 training batches with batch size 8). The staging cell shows a byte-level progress bar separately for each chunk; the training bar reports losses, CUDA memory, local sample count, and data wait per batch. The manifest, checkpoint, history, and metrics persist on Drive. The copied raw files are in temporary `/content` storage and must be staged again after a runtime reset.

In [ ]:
# 11. Locked test evaluation and qualitative depth/normal inspection
test_metrics=evaluate(model,loaders['test'])
print(json.dumps(test_metrics,indent=2))
metrics_path=OUTPUT_DIR/'test_metrics.json'; metrics_tmp=metrics_path.with_name(metrics_path.name+'.tmp')
metrics_tmp.write_text(json.dumps(test_metrics,indent=2)); metrics_tmp.replace(metrics_path)
model.eval(); x,d,n,m=next(iter(loaders['test'])); x=x.to(DEVICE)
with torch.inference_mode(): pd,pn=model(x)
rgb=((x[0].cpu()*torch.tensor([.229,.224,.225])[:,None,None])+torch.tensor([.485,.456,.406])[:,None,None]).permute(1,2,0).clamp(0,1)
fig,ax=plt.subplots(2,3,figsize=(12,7))
ax[0,0].imshow(rgb); ax[0,0].set_title('Input RGB')
ax[0,1].imshow(d[0,0],cmap='viridis'); ax[0,1].set_title('Depth GT (m)')
ax[0,2].imshow(n[0].permute(1,2,0)*.5+.5); ax[0,2].set_title('Normals GT')
ax[1,0].imshow(pd[0,0].cpu(),cmap='viridis'); ax[1,0].set_title('Depth prediction (m)')
ax[1,1].imshow(pn[0].cpu().permute(1,2,0)*.5+.5); ax[1,1].set_title('Normals prediction')
ax[1,2].imshow(m[0,0],cmap='gray'); ax[1,2].set_title('Valid mask')
[z.axis('off') for z in ax.flat]; plt.tight_layout(); plt.show()


In [ ]:
# 12. Optional fixed-shape Core ML export; run with a pinned coremltools version.
# Conversion success is not a performance result. Follow with Xcode performance report on target iPhone(s).
EXPORT_COREML=False
if EXPORT_COREML:
    import coremltools as ct
    class ExportWrapper(nn.Module):
        def __init__(self,base): super().__init__(); self.base=base
        def forward(self,x): return self.base(x)
    export_model=ExportWrapper(model.eval()).cpu().eval()
    example=torch.zeros(1,3,*INPUT_SIZE)
    traced=torch.jit.trace(export_model,example,strict=True)
    # The model takes ImageNet-normalized RGB tensors; export as tensor input and normalize in app preprocessing.
    mlmodel=ct.convert(traced,source='pytorch',inputs=[ct.TensorType(name='image',shape=example.shape)],outputs=[ct.TensorType(name='depth'),ct.TensorType(name='normals')],convert_to='mlprogram',minimum_deployment_target=ct.target.iOS16,compute_precision=ct.precision.FLOAT16)
    mlmodel.author='Apple Capstone Portrait Relighting'; mlmodel.short_description='Portrait depth and surface-normal research model; evaluate on device before use.'
    out=OUTPUT_DIR/'portrait_geometry_depth_normals.mlpackage'; mlmodel.save(str(out)); print('Saved',out)
    # Numerical parity check on a representative fixed-shape tensor.
    if platform.system() == 'Darwin':
        with torch.inference_mode(): torch_out=export_model(example)
        coreml_out=mlmodel.predict({'image':example.numpy()})
        for name,ref in zip(('depth','normals'),torch_out):
            delta=np.abs(coreml_out[name]-ref.numpy())
            print(name,'max_abs_error',float(delta.max()),'mean_abs_error',float(delta.mean()))
            if not np.isfinite(delta).all(): raise FloatingPointError(f'Non-finite Core ML output for {name}')
    else:
        print('Export completed. Core ML prediction/parity check requires macOS; run that check on a Mac or target device.')

# Device evaluation checklist:
# 1) Xcode Model Performance Report: operation placement (ANE/GPU/CPU), load and prediction duration.
# 2) On each target iPhone/iOS: warm-up; >=200 predictions; report p50/p95, cold load, package size, peak memory.
# 3) Measure Vision segmentation + preprocessing + model + relighting + compositing frame time; include thermal steady state.
# 4) Compare FP32/FP16/quantized candidates for parity, quality, model size, and device latency.
# 5) 60 FPS requires the complete pipeline to sustain <16.7 ms/frame with acceptable quality and thermal behavior.

## Interpretation and required experiments
## Backbone evidence and decision
- **Keep FastViT-T8 as the baseline.** Apple reports FastViT-T8 at 3.6M parameters and provides Core ML classification measurements on iPhone models (for example 0.52 ms on iPhone 16 Pro). The FastViT paper and official model page support mobile efficiency and Core ML availability. Those figures are for the backbone/classifier model, not our 256×256 two-output dense model.
- **MobileNetV3-Large is a useful ablation.** Its paper targets mobile CPU efficiency, reports about 5.4M parameters, and introduces LR-ASPP for dense prediction. Its published CPU measurements use Pixel devices; they do not prove better Apple Neural Engine performance.
- **These published latency values are not directly comparable.** The devices, runtime, input graph, precision, and task differ. Change only the backbone while holding the dataset split, FPN/heads, resolution, training schedule, and Core ML conversion constant. Select based on held-out task quality, model size, and Xcode measurements on the intended iPhone.

- First establish that label overlays and coordinate conventions are correct. Pseudo-label metrics quantify agreement with the teacher, not necessarily physical truth.
- Ablate: (A) FastViT-T8 baseline with compact FPN, (B) MobileNetV3-Large comparison using the same FPN/heads, (C) fixed versus uncertainty-weighted losses. Change only the backbone for the backbone ablation; compare validation metrics, parameter count, converted model size, and on-device latency. Published backbone-only classification timings are not comparable to this dense prediction graph. - Compare against task-appropriate baselines: teacher outputs for geometry, constant/median depth where meaningful, 
- For the product claim (relighting), build a controlled capture set with fixed subject, camera/exposure/white balance, and measured light positions/intensities. Hold out identities and lighting configurations. Evaluate relit RGB against captures and inspect temporal stability on video.
- References: [FastViT paper](https://arxiv.org/abs/2303.14189); [Sapiens paper](https://arxiv.org/abs/2408.12569); [uncertainty weighting](https://arxiv.org/abs/1705.07115); [Core ML performance report](https://developer.apple.com/documentation/coreml/analyzing-a-core-ml-model-s-performance-in-xcode); [Core ML Tools PyTorch conversion](https://apple.github.io/coremltools/docs-guides/source/convert-pytorch.html); [MobileNetV3 paper](https://openaccess.thecvf.com/content_ICCV_2019/html/Howard_Searching_for_MobileNetV3_ICCV_2019_paper.html).
## Dataset trial scope
- The automatic Colab trial uses the official Microsoft SynthHuman/DAViD download utility. Its synthetic sample labels include RGB, absolute depth in centimetres, XYZ normals, and alpha masks; the active experiment supervises depth and normals only.
- The official release offers a single-sample smoke test and a single chunk (up to 5,000 samples; up to 8.75 GB); the full dataset is about 330 GB. The trial split uses sample IDs because the distributed files do not expose identity IDs, so it is not an identity-disjoint generalization test.
